Import packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("train_FD004_2.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
print(df.isna().sum().sum(), "missing values")
print(df.duplicated().sum(), "duplicates")

In [ ]:
print(df.unit.nunique(),'Unique engine are Working')

Engine life-cycle analysis

In [ ]:
unit_life = df.groupby("unit")["cycle"].agg(
    first_cycle="min", last_cycle="max", observations="count"
)
display(unit_life.describe())
print("Engines not starting at cycle 1:", (unit_life["first_cycle"] != 1).sum())
print("Engines where max cycle != observation count:",
      (unit_life["last_cycle"] != unit_life["observations"]).sum())

plt.figure(figsize=(10, 5))
sns.histplot(unit_life["last_cycle"], bins=30, kde=True)
plt.title("Distribution of engine lifespans")
plt.xlabel("Last observed cycle")
plt.ylabel("Number of engines")
plt.tight_layout()
plt.show()


Sensor distributions and variability

In [ ]:
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
sensor_summary = df[sensor_cols].agg(["mean", "std", "min", "max", "nunique"]).T
display(sensor_summary.sort_values("std"))

sensor_summary["std"].sort_values().plot(kind="bar", figsize=(13, 5))
plt.title("Standard deviation by sensor")
plt.ylabel("Standard deviation")
plt.tight_layout()
plt.show()

# Histograms for all sensors
df[sensor_cols].hist(figsize=(18, 16), bins=35)
plt.suptitle("Sensor distributions", y=1.02)
plt.tight_layout()
plt.show()


Operating settings and sensor relationships

In [ ]:
op_cols = ["op_setting_1", "op_setting_2", "op_setting_3"]
print("Unique operating-setting combinations:",
      df[op_cols].drop_duplicates().shape[0])

plt.figure(figsize=(8, 6))
sns.heatmap(df[sensor_cols + op_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation matrix: operating settings and sensors")
plt.tight_layout()
plt.show()


Create RUL target

In [ ]:
#RUL = max_cycle_for_that_unit - current_cycle
df["RUL"] = df.groupby("unit")["cycle"].transform("max") - df["cycle"]
df["RUL_capped_125"] = df["RUL"].clip(upper=125)

display(df[["unit", "cycle", "RUL", "RUL_capped_125"]].head(10))
display(df["RUL"].describe())

plt.figure(figsize=(9, 5))
sns.histplot(df["RUL"], bins=40, kde=True)
plt.title("Training RUL distribution")
plt.xlabel("RUL (cycles)")
plt.tight_layout()
plt.show()


Temporal sensor feature engineering
For each sensor, create:
- lag 1 and lag 2 values
- one-cycle difference
- rolling mean over 5 and 10 cycles
- rolling standard deviation over 10 cycles
Features are calculated separately within each engine, after sorting by cycle. Rolling windows include the current cycle, which is available when predicting at the end of that cycle. Initial lag/difference values are filled at the engine boundary; rolling standard deviation for the first observation is set to zero.

In [ ]:
df = df.sort_values(["unit", "cycle"]).reset_index(drop=True)
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

for sensor in sensor_cols:
    grouped = df.groupby("unit", sort=False)[sensor]
    df[f"{sensor}_lag1"] = grouped.shift(1)
    df[f"{sensor}_lag2"] = grouped.shift(2)
    df[f"{sensor}_diff1"] = grouped.diff(1)
    df[f"{sensor}_roll_mean5"] = grouped.transform(
        lambda s: s.rolling(5, min_periods=1).mean()
    )
    df[f"{sensor}_roll_mean10"] = grouped.transform(
        lambda s: s.rolling(10, min_periods=1).mean()
    )
    df[f"{sensor}_roll_std10"] = grouped.transform(
        lambda s: s.rolling(10, min_periods=2).std()
    )

for c in [c for c in df.columns if c.endswith(("_lag1", "_lag2"))]:
    sensor = c.rsplit("_lag", 1)[0]
    df[c] = df[c].fillna(df[sensor])

for c in [c for c in df.columns if c.endswith("_diff1")]:
    df[c] = df[c].fillna(0)

std_cols = [c for c in df.columns if c.endswith("_roll_std10")]
df[std_cols] = df[std_cols].fillna(0)

print("Engineered shape:", df.shape)
print("Remaining missing values:", int(df.isna().sum().sum()))
display(df.head())


Export engineered training data:   
The resulting file includes the original columns, RUL, RUL_capped_125, and the temporal sensor features.

For model training, split by engine unit, not random rows, to avoid the same engine appearing in both train and validation. Fit scalers/encoders on the training split only, ideally inside an sklearn Pipeline.

In [ ]:
OUTPUT_PATH = "train_FD004_feature_engineered.csv"
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
